# ETL_dimensoes

Creates the dimension tables, final fact tables, and consumption views for FastAPI.

**Note:** the `exame` field was removed from the final model because it will not be used in the temporal/spatial dashboard.

## 1. Configuration and loading of transformed tables

This notebook assumes that the source ETL notebooks have already created the intermediate tables in Databricks.

In [0]:
CATALOG = "workspace"
SCHEMA = "default"
TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"

FACT_ARBOVIROSES = f"{TABLE_PREFIX}.fato_arboviroses"
FACT_COVID = f"{TABLE_PREFIX}.fato_covid"
FACT_ESCORPIAO = f"{TABLE_PREFIX}.fato_escorpiao"
FACT_DENGUE_SPATIAL = f"{TABLE_PREFIX}.fato_dengue_espacial"
FACT_CHIK_SPATIAL = f"{TABLE_PREFIX}.fato_chikungunya_espacial"

df_fato_arboviroses = spark.table(FACT_ARBOVIROSES)
df_fato_covid = spark.table(FACT_COVID)
df_fato_escorpiao = spark.table(FACT_ESCORPIAO)
df_fato_dengue_espacial = spark.table(FACT_DENGUE_SPATIAL)
df_fato_chikungunya_espacial = spark.table(FACT_CHIK_SPATIAL)

print("Intermediate tables loaded successfully.")

## 2. Helper functions

In [0]:
from functools import reduce
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, lit, lower, upper, trim, row_number, concat_ws, lpad,
    create_map, element_at, year, month, regexp_replace, translate,
    count as spark_count, sum as spark_sum
)
from pyspark.sql.types import StringType, DoubleType, StructType, StructField

MONTH_ORDER = {
    "JANEIRO": 1,
    "FEVEREIRO": 2,
    "MARÇO": 3,
    "ABRIL": 4,
    "MAIO": 5,
    "JUNHO": 6,
    "JULHO": 7,
    "AGOSTO": 8,
    "SETEMBRO": 9,
    "OUTUBRO": 10,
    "NOVEMBRO": 11,
    "DEZEMBRO": 12,
}

ORDER_MONTH = {value: key for key, value in MONTH_ORDER.items()}
order_month_map = create_map([lit(x) for pair in ORDER_MONTH.items() for x in pair])

ACCENTED = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇáàâãäéèêëíìîïóòôõöúùûüç"
UNACCENTED = "AAAAAEEEEIIIIOOOOOUUUUCaaaaaeeeeiiiiooooouuuuc"


def normalize_key(column):
    """Normalize text keys: lowercase, no accents, and no duplicated spaces."""
    return regexp_replace(
        translate(lower(trim(column.cast("string"))), ACCENTED, UNACCENTED),
        r"\s+",
        " "
    )


def ensure_column(df, column_name, data_type="string"):
    """Ensure that a column exists, creating it as null when necessary."""
    if column_name in df.columns:
        return df
    return df.withColumn(column_name, lit(None).cast(data_type))


def with_normalized_disease(df, default_disease=None):
    """Ensure a normalized doenca column. Uses default_disease if the table does not contain this column."""
    if "doenca" in df.columns:
        return df.withColumn("doenca", normalize_key(col("doenca")))
    return df.withColumn("doenca", lit(default_disease))

## 3. Create `dim_tempo`

The time dimension is monthly. Dengue spatial records are linked to this dimension through their record date.

In [0]:
df_time_from_temporal = (
    df_fato_arboviroses.select("ano", "mes", "ordem_mes")
    .unionByName(df_fato_covid.select("ano", "mes", "ordem_mes"))
    .unionByName(df_fato_escorpiao.select("ano", "mes", "ordem_mes"))
)

df_time_from_dengue_spatial = (
    df_fato_dengue_espacial
    .filter(col("data").isNotNull())
    .select(
        year(col("data")).alias("ano"),
        element_at(order_month_map, month(col("data"))).alias("mes"),
        month(col("data")).alias("ordem_mes")
    )
)

df_dim_tempo = (
    df_time_from_temporal
    .unionByName(df_time_from_dengue_spatial)
    .filter(col("ano").isNotNull())
    .filter(col("ordem_mes").isNotNull())
    .dropDuplicates()
    .withColumn(
        "id_tempo",
        concat_ws(
            "",
            col("ano").cast("string"),
            lpad(col("ordem_mes").cast("string"), 2, "0")
        ).cast("int")
    )
    .withColumn(
        "ano_mes",
        concat_ws(
            "-",
            col("ano").cast("string"),
            lpad(col("ordem_mes").cast("string"), 2, "0")
        )
    )
    .select("id_tempo", "ano", "mes", "ordem_mes", "ano_mes")
    .orderBy("ano", "ordem_mes")
)

df_dim_tempo.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{TABLE_PREFIX}.dim_tempo")

display(df_dim_tempo)

## 4. Create `dim_doenca`

In [0]:
disease_frames = [
    with_normalized_disease(df_fato_arboviroses).select("doenca"),
    with_normalized_disease(df_fato_covid).select("doenca"),
    with_normalized_disease(df_fato_escorpiao, "escorpiao").select("doenca"),
    with_normalized_disease(df_fato_dengue_espacial).select("doenca"),
    with_normalized_disease(df_fato_chikungunya_espacial).select("doenca"),
]

df_doencas_base = (
    reduce(lambda a, b: a.unionByName(b), disease_frames)
    .filter(col("doenca").isNotNull())
    .filter(col("doenca") != "")
    .dropDuplicates()
)

df_dim_doenca = (
    df_doencas_base
    .withColumn("id_doenca", row_number().over(Window.orderBy("doenca")))
    .select("id_doenca", "doenca")
    .orderBy("id_doenca")
)

df_dim_doenca.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{TABLE_PREFIX}.dim_doenca")

display(df_dim_doenca)

## 5. Create `dim_bairro`

The neighborhood dimension keeps only the identifier and the normalized neighborhood name. Geolocation is handled in a separate notebook because it depends on an external/manual source of coordinates or GeoJSON.

In [0]:
df_bairros_base = (
    df_fato_dengue_espacial.select(normalize_key(col("bairro")).alias("bairro"))
    .unionByName(df_fato_chikungunya_espacial.select(normalize_key(col("bairro")).alias("bairro")))
    .filter(col("bairro").isNotNull())
    .filter(col("bairro") != "")
    .dropDuplicates()
)

df_dim_bairro = (
    df_bairros_base
    .withColumn("id_bairro", row_number().over(Window.orderBy("bairro")))
    .select("id_bairro", "bairro")
    .orderBy("id_bairro")
)

df_dim_bairro.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{TABLE_PREFIX}.dim_bairro")

display(df_dim_bairro)

## 6. Create `dim_sexo`

In [0]:
df_sexo_base = (
    df_fato_dengue_espacial.select(normalize_key(col("sexo")).alias("sexo"))
    .unionByName(df_fato_chikungunya_espacial.select(normalize_key(col("sexo")).alias("sexo")))
    .filter(col("sexo").isNotNull())
    .filter(col("sexo") != "")
    .dropDuplicates()
)

df_dim_sexo = (
    df_sexo_base
    .withColumn("id_sexo", row_number().over(Window.orderBy("sexo")))
    .select("id_sexo", "sexo")
    .orderBy("id_sexo")
)

df_dim_sexo.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{TABLE_PREFIX}.dim_sexo")

display(df_dim_sexo)

## 7. Reload dimension tables

In [0]:
df_dim_tempo = spark.table(f"{TABLE_PREFIX}.dim_tempo")
df_dim_doenca = spark.table(f"{TABLE_PREFIX}.dim_doenca")
df_dim_bairro = spark.table(f"{TABLE_PREFIX}.dim_bairro")
df_dim_sexo = spark.table(f"{TABLE_PREFIX}.dim_sexo")

## 8. Model final temporal fact tables

In [0]:
def model_temporal_fact(df_fact, output_table, default_disease=None):
    df_prepared = with_normalized_disease(df_fact, default_disease)

    df_final = (
        df_prepared
        .join(df_dim_tempo, on=["ano", "mes", "ordem_mes"], how="left")
        .join(df_dim_doenca, on="doenca", how="left")
        .select("id_tempo", "id_doenca", "casos", "obitos")
    )

    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(output_table)

    return df_final


df_fato_arboviroses_final = model_temporal_fact(
    df_fato_arboviroses,
    f"{TABLE_PREFIX}.fato_arboviroses_final"
)

df_fato_covid_final = model_temporal_fact(
    df_fato_covid,
    f"{TABLE_PREFIX}.fato_covid_final"
)

df_fato_escorpiao_final = model_temporal_fact(
    df_fato_escorpiao,
    f"{TABLE_PREFIX}.fato_escorpiao_final",
    default_disease="escorpiao"
)

display(df_fato_arboviroses_final)
display(df_fato_covid_final)
display(df_fato_escorpiao_final)

## 9. Model final dengue spatial fact table

The `exame` field is not included in the final table.

In [0]:
df_dengue_spatial_base = ensure_column(df_fato_dengue_espacial, "unidade_notificacao", "string")

df_dengue_spatial_time = (
    df_dengue_spatial_base
    .withColumn("ano", year(col("data")))
    .withColumn("ordem_mes", month(col("data")))
    .withColumn("mes", element_at(order_month_map, col("ordem_mes")))
)

df_fato_dengue_espacial_final = (
    df_dengue_spatial_time
    .withColumn("doenca", normalize_key(col("doenca")))
    .withColumn("bairro", normalize_key(col("bairro")))
    .withColumn("sexo", normalize_key(col("sexo")))
    .join(df_dim_doenca, on="doenca", how="left")
    .join(df_dim_bairro, on="bairro", how="left")
    .join(df_dim_sexo, on="sexo", how="left")
    .join(df_dim_tempo, on=["ano", "mes", "ordem_mes"], how="left")
    .select(
        "id_doenca",
        "id_bairro",
        "id_sexo",
        "id_tempo",
        "data",
        "idade",
        "unidade_notificacao"
    )
)

df_fato_dengue_espacial_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{TABLE_PREFIX}.fato_dengue_espacial_final")

display(df_fato_dengue_espacial_final)

## 10. Model final chikungunya spatial fact table

The chikungunya spatial dataset does not contain daily dates; therefore, it keeps `ano_referencia`.

In [0]:
df_chikungunya_spatial_base = df_fato_chikungunya_espacial

if "ano_referencia" not in df_chikungunya_spatial_base.columns:
    df_chikungunya_spatial_base = df_chikungunya_spatial_base.withColumn("ano_referencia", lit(2026))

df_fato_chikungunya_espacial_final = (
    df_chikungunya_spatial_base
    .withColumn("doenca", normalize_key(col("doenca")))
    .withColumn("bairro", normalize_key(col("bairro")))
    .withColumn("sexo", normalize_key(col("sexo")))
    .join(df_dim_doenca, on="doenca", how="left")
    .join(df_dim_bairro, on="bairro", how="left")
    .join(df_dim_sexo, on="sexo", how="left")
    .select(
        "id_doenca",
        "id_bairro",
        "id_sexo",
        "ano_referencia",
        "idade"
    )
)

df_fato_chikungunya_espacial_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{TABLE_PREFIX}.fato_chikungunya_espacial_final")

display(df_fato_chikungunya_espacial_final)

## 11. Create views for FastAPI consumption

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {TABLE_PREFIX}.vw_serie_temporal AS
SELECT
    'arboviroses' AS fonte,
    t.ano,
    t.mes,
    t.ordem_mes,
    t.ano_mes,
    d.doenca,
    f.casos,
    f.obitos
FROM {TABLE_PREFIX}.fato_arboviroses_final f
LEFT JOIN {TABLE_PREFIX}.dim_tempo t ON f.id_tempo = t.id_tempo
LEFT JOIN {TABLE_PREFIX}.dim_doenca d ON f.id_doenca = d.id_doenca

UNION ALL

SELECT
    'covid' AS fonte,
    t.ano,
    t.mes,
    t.ordem_mes,
    t.ano_mes,
    d.doenca,
    f.casos,
    f.obitos
FROM {TABLE_PREFIX}.fato_covid_final f
LEFT JOIN {TABLE_PREFIX}.dim_tempo t ON f.id_tempo = t.id_tempo
LEFT JOIN {TABLE_PREFIX}.dim_doenca d ON f.id_doenca = d.id_doenca

UNION ALL

SELECT
    'escorpiao' AS fonte,
    t.ano,
    t.mes,
    t.ordem_mes,
    t.ano_mes,
    d.doenca,
    f.casos,
    f.obitos
FROM {TABLE_PREFIX}.fato_escorpiao_final f
LEFT JOIN {TABLE_PREFIX}.dim_tempo t ON f.id_tempo = t.id_tempo
LEFT JOIN {TABLE_PREFIX}.dim_doenca d ON f.id_doenca = d.id_doenca
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {TABLE_PREFIX}.vw_casos_espaciais AS
SELECT
    'dengue_espacial' AS fonte,
    d.doenca,
    b.bairro,
    s.sexo,
    t.ano,
    t.mes,
    t.ordem_mes,
    t.ano_mes,
    f.data,
    f.idade,
    f.unidade_notificacao
FROM {TABLE_PREFIX}.fato_dengue_espacial_final f
LEFT JOIN {TABLE_PREFIX}.dim_doenca d ON f.id_doenca = d.id_doenca
LEFT JOIN {TABLE_PREFIX}.dim_bairro b ON f.id_bairro = b.id_bairro
LEFT JOIN {TABLE_PREFIX}.dim_sexo s ON f.id_sexo = s.id_sexo
LEFT JOIN {TABLE_PREFIX}.dim_tempo t ON f.id_tempo = t.id_tempo

UNION ALL

SELECT
    'chikungunya_espacial' AS fonte,
    d.doenca,
    b.bairro,
    s.sexo,
    f.ano_referencia AS ano,
    CAST(NULL AS STRING) AS mes,
    CAST(NULL AS INT) AS ordem_mes,
    CAST(NULL AS STRING) AS ano_mes,
    CAST(NULL AS DATE) AS data,
    f.idade,
    CAST(NULL AS STRING) AS unidade_notificacao
FROM {TABLE_PREFIX}.fato_chikungunya_espacial_final f
LEFT JOIN {TABLE_PREFIX}.dim_doenca d ON f.id_doenca = d.id_doenca
LEFT JOIN {TABLE_PREFIX}.dim_bairro b ON f.id_bairro = b.id_bairro
LEFT JOIN {TABLE_PREFIX}.dim_sexo s ON f.id_sexo = s.id_sexo
""")

print("Views created:")
print(f"{TABLE_PREFIX}.vw_serie_temporal")
print(f"{TABLE_PREFIX}.vw_casos_espaciais")

## 12. Final model validation

These validations are kept in this notebook because they check the final tables created here.

In [0]:
FAIL_ON_VALIDATION_ERROR = True

final_tables = [
    "dim_tempo",
    "dim_doenca",
    "dim_bairro",
    "dim_sexo",
    "fato_arboviroses_final",
    "fato_covid_final",
    "fato_escorpiao_final",
    "fato_dengue_espacial_final",
    "fato_chikungunya_espacial_final",
    "vw_serie_temporal",
    "vw_casos_espaciais",
]

print("=== FINAL TABLE AND VIEW ROW COUNTS ===")
for table_name in final_tables:
    count_rows = spark.table(f"{TABLE_PREFIX}.{table_name}").count()
    print(f"{table_name}: {count_rows} rows")

print("\n=== VALIDATION: EXAME FIELD REMOVED ===")
for table_name in ["fato_dengue_espacial_final", "vw_casos_espaciais"]:
    cols = spark.table(f"{TABLE_PREFIX}.{table_name}").columns
    print(f"{table_name}: {cols}")
    if "exame" in cols:
        raise ValueError(f"ERROR: field 'exame' still exists in {table_name}")

validation_checks = {
    "fato_arboviroses_final": ["id_tempo", "id_doenca"],
    "fato_covid_final": ["id_tempo", "id_doenca"],
    "fato_escorpiao_final": ["id_tempo", "id_doenca"],
    "fato_dengue_espacial_final": ["id_doenca", "id_bairro", "id_sexo", "id_tempo"],
    "fato_chikungunya_espacial_final": ["id_doenca", "id_bairro", "id_sexo"],
}

has_validation_error = False

print("\n=== VALIDATION: NULL KEYS IN FACT TABLES ===")
for table_name, key_columns in validation_checks.items():
    print(f"\n--- {table_name} ---")
    df_check = spark.table(f"{TABLE_PREFIX}.{table_name}")

    null_summary = df_check.select([
        spark_sum(col(c).isNull().cast("int")).alias(f"{c}_nulls")
        for c in key_columns
    ])

    display(null_summary)

    row = null_summary.collect()[0].asDict()
    if any(value and value > 0 for value in row.values()):
        has_validation_error = True
        print("Problematic records:")
        condition = " OR ".join([f"{c} IS NULL" for c in key_columns])
        display(df_check.filter(condition))

print("\n=== VALIDATION: ORIGINAL VS FINAL ROW COUNTS ===")
row_count_checks = [
    ("arboviroses", df_fato_arboviroses, spark.table(f"{TABLE_PREFIX}.fato_arboviroses_final")),
    ("covid", df_fato_covid, spark.table(f"{TABLE_PREFIX}.fato_covid_final")),
    ("escorpiao", df_fato_escorpiao, spark.table(f"{TABLE_PREFIX}.fato_escorpiao_final")),
    ("dengue_espacial", df_fato_dengue_espacial, spark.table(f"{TABLE_PREFIX}.fato_dengue_espacial_final")),
    ("chikungunya_espacial", df_fato_chikungunya_espacial, spark.table(f"{TABLE_PREFIX}.fato_chikungunya_espacial_final")),
]

for label, original_df, final_df in row_count_checks:
    original_count = original_df.count()
    final_count = final_df.count()
    print(f"{label}: original={original_count} | final={final_count}")
    if original_count != final_count:
        has_validation_error = True

if has_validation_error and FAIL_ON_VALIDATION_ERROR:
    raise ValueError("Validation found problems. Check the tables displayed above.")

print("\nValidation completed.")

## 13. Smoke tests for the API views

In [0]:
print("=== SAMPLE: vw_serie_temporal ===")
display(
    spark.sql(f"""
        SELECT
            doenca,
            ano,
            mes,
            ordem_mes,
            SUM(casos) AS casos,
            SUM(obitos) AS obitos
        FROM {TABLE_PREFIX}.vw_serie_temporal
        GROUP BY doenca, ano, mes, ordem_mes
        ORDER BY doenca, ano, ordem_mes
    """)
)

print("=== SAMPLE: vw_casos_espaciais ===")
display(
    spark.sql(f"""
        SELECT
            fonte,
            doenca,
            bairro,
            sexo,
            ano,
            COUNT(*) AS records
        FROM {TABLE_PREFIX}.vw_casos_espaciais
        GROUP BY fonte, doenca, bairro, sexo, ano
        ORDER BY fonte, doenca, bairro, sexo
    """)
)